# E2.1 — Repair (Capping) vs CDP Constraint Handling

Compares implicit constraint repair (production capping) with explicit constraint handling (constrained dominance) in terms of convergence and diversity of the Pareto front.

**Experiment:** 2 constraint handling strategies × N seeds, using the best encoding from E1.1.

| Config | Strategy |
|--------|----------|
| capping | FCFS+pro-rata production capping (repair) — all solutions feasible |
| cdp | No capping + Constrained Dominance Principle (Deb, 2002) |

**Metrics:** HVI, IGD, convergence curves, feasibility rate (CDP only).

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import wilcoxon

def paired_wilcoxon(metric_a, seeds_a, metric_b, seeds_b):
    """Wilcoxon signed-rank paired by seed. Returns (p_value, n_common)."""
    seed_to_a = dict(zip(seeds_a, metric_a))
    seed_to_b = dict(zip(seeds_b, metric_b))
    common = sorted(set(seed_to_a) & set(seed_to_b))
    va = np.array([seed_to_a[s] for s in common])
    vb = np.array([seed_to_b[s] for s in common])
    if np.allclose(va, vb):
        return 1.0, len(common)
    return wilcoxon(va, vb, alternative="two-sided").pvalue, len(common)


def paired_diff(metric_a, seeds_a, metric_b, seeds_b):
    """Per-seed (A - B) differences, aligned by seed."""
    sa_to_v = dict(zip(seeds_a, metric_a))
    sb_to_v = dict(zip(seeds_b, metric_b))
    common = sorted(set(sa_to_v) & set(sb_to_v))
    return np.array([sa_to_v[s] - sb_to_v[s] for s in common])

from src.experiments.runner import load_experiment_results, load_run_result
from src.experiments.metrics import (
    hypervolume_2d,
    inverted_generational_distance,
    construct_reference_front,
)

config_order = ["capping", "cdp"]
config_labels = ["Capping (Repair)", "CDP (Constrained Dominance)"]
colors = ["steelblue", "coral"]

## 1. Load Results

In [ ]:
runs = load_experiment_results("E2.1")
print(f"Loaded {len(runs)} runs")

configs = defaultdict(list)
for run in runs:
    s = run["summary"]
    if s["status"] != "completed":
        print(f"  SKIPPING failed run: {s['run_id']}")
        continue
    configs[s["config_name"]].append(run)

for name, run_list in sorted(configs.items()):
    times = [r["summary"]["elapsed_seconds"] for r in run_list]
    print(f"  {name:20s}: {len(run_list)} runs, avg time {np.mean(times):.1f}s")

## 2. Extract Pareto Fronts and Compute Metrics

In [ ]:
fronts_by_config = {}
for name, run_list in configs.items():
    fronts = []
    for run in run_list:
        result = load_run_result(run["pickle_path"])
        pf = result["pareto_front"]
        if pf:
            obj = np.array([e.get("objectives_final", e["objectives"]) for e in pf])
            fronts.append(obj)
        else:
            fronts.append(np.empty((0, 2)))
    fronts_by_config[name] = fronts

all_objectives = [f for fronts in fronts_by_config.values() for f in fronts if len(f) > 0]
reference_front = construct_reference_front(all_objectives)
print(f"Reference front: {len(reference_front)} non-dominated solutions")

all_combined = np.vstack(all_objectives)
ref_point = np.array([
    all_combined[:, 0].min() - abs(all_combined[:, 0].min()) * 0.1,
    all_combined[:, 1].min() - abs(all_combined[:, 1].min()) * 0.1,
])

seeds_by_config = {name: [r["summary"]["seed"] for r in run_list] for name, run_list in configs.items()}
hvi_by_config = {}
igd_by_config = {}
for name, fronts in fronts_by_config.items():
    hvi_by_config[name] = np.array([hypervolume_2d(f, ref_point) for f in fronts])
    igd_by_config[name] = np.array([inverted_generational_distance(f, reference_front) for f in fronts])

print(f"\n{'Config':20s} {'HVI mean':>12s} {'HVI std':>10s} {'IGD mean':>12s} {'IGD std':>10s}")
print("-" * 66)
for name in config_order:
    if name not in hvi_by_config: continue
    hvi, igd = hvi_by_config[name], igd_by_config[name]
    print(f"{name:20s} {hvi.mean():12.1f} {hvi.std():10.1f} {igd.mean():12.4f} {igd.std():10.4f}")

## 3. Box Plots — HVI and IGD

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

active = [c for c in config_order if c in hvi_by_config]
labels = [config_labels[config_order.index(c)] for c in active]

for ax, data_dict, ylabel, title in [
    (axes[0], {c: hvi_by_config[c] / 1e6 for c in active}, r"HVI ($\times 10^6$)", "Hypervolume Indicator (higher = better)"),
    (axes[1], igd_by_config, "IGD", "Inverted Generational Distance (lower = better)"),
]:
    data = [data_dict[c] for c in active]
    bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6)
    for p, color in zip(bp["boxes"], colors):
        p.set_facecolor(color); p.set_alpha(0.7)
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0.3, len(active) + 0.7)

plt.tight_layout()
plt.show()

## 4. Statistical Significance

In [ ]:
active = [c for c in config_order if c in hvi_by_config]
if len(active) == 2:
    c1, c2 = active
    p_hvi, n_pair = paired_wilcoxon(hvi_by_config[c1], seeds_by_config[c1], hvi_by_config[c2], seeds_by_config[c2])
    p_igd, _ = paired_wilcoxon(igd_by_config[c1], seeds_by_config[c1], igd_by_config[c2], seeds_by_config[c2])

    l1 = config_labels[config_order.index(c1)]
    l2 = config_labels[config_order.index(c2)]
    print(f"Wilcoxon signed-rank, paired by seed (n = {n_pair})")
    print(f"{l1} vs {l2}:")
    print(f"  HVI: p = {p_hvi:.2e} {'***' if p_hvi < 0.001 else '**' if p_hvi < 0.01 else '*' if p_hvi < 0.05 else 'n.s.'}")
    print(f"  IGD: p = {p_igd:.2e} {'***' if p_igd < 0.001 else '**' if p_igd < 0.01 else '*' if p_igd < 0.05 else 'n.s.'}")

    # Effect size: median per-seed difference (A - B)
    hvi_diff = paired_diff(hvi_by_config[c1], seeds_by_config[c1], hvi_by_config[c2], seeds_by_config[c2])
    igd_diff = paired_diff(igd_by_config[c1], seeds_by_config[c1], igd_by_config[c2], seeds_by_config[c2])
    print(f"  HVI median per-seed diff ({l1} - {l2}): {np.median(hvi_diff):+.2e}")
    print(f"  IGD median per-seed diff ({l1} - {l2}): {np.median(igd_diff):+.2e}")
else:
    print("Need exactly 2 configs for pairwise comparison")


## 5. Overlaid Pareto Fronts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

for i, name in enumerate(config_order):
    if name not in hvi_by_config: continue
    hvi_vals = hvi_by_config[name]
    median_idx = np.argmin(np.abs(hvi_vals - np.median(hvi_vals)))
    front = fronts_by_config[name][median_idx]
    if len(front) == 0: continue
    label = f"{config_labels[i]} (n = {len(front)})"
    ax.scatter(front[:, 1], front[:, 0], color=colors[i], label=label,
               s=70, alpha=0.75, edgecolor="black", linewidth=0.5)

ax.set_xlabel("CVaR (MNOK)"); ax.set_ylabel("Expected ROV (MNOK)")
ax.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()

## 6. Convergence Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, name in enumerate(config_order):
    if name not in configs: continue
    all_rov, all_cvar = [], []
    for run in configs[name]:
        hp = run["run_dir"] / "history.pkl"
        if hp.exists():
            with open(hp, "rb") as f: h = pickle.load(f)
            all_rov.append(h["best_rov"]); all_cvar.append(h["best_cvar"])
    if not all_rov: continue

    max_len = max(len(r) for r in all_rov)
    rov_arr = np.full((len(all_rov), max_len), np.nan)
    cvar_arr = np.full((len(all_cvar), max_len), np.nan)
    for j, (r, c) in enumerate(zip(all_rov, all_cvar)):
        rov_arr[j, :len(r)] = r; cvar_arr[j, :len(c)] = c

    iters = np.arange(1, max_len + 1)
    axes[0].plot(iters, np.nanmean(rov_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)
    axes[1].plot(iters, np.nanmean(cvar_arr, axis=0), color=colors[i], label=config_labels[i], alpha=0.8)

axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("Best E[ROV] (MNOK)")
axes[0].set_title("Convergence — Best Expected ROV"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("Iteration"); axes[1].set_ylabel("Best CVaR (MNOK)")
axes[1].set_title("Convergence — Best CVaR"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary and CHT Selection

Selects the constraint handling technique carried forward to E3.1. If results are comparable, capping is preferred because it is simpler.

In [ ]:
active = [c for c in config_order if c in hvi_by_config]
print("Ranking by mean HVI (higher = better):")
for rank, (name, val) in enumerate(sorted([(n, hvi_by_config[n].mean()) for n in active], key=lambda x: -x[1]), 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:30s} HVI = {val:.1f}")

print("\nRanking by mean IGD (lower = better):")
for rank, (name, val) in enumerate(sorted([(n, igd_by_config[n].mean()) for n in active], key=lambda x: x[1]), 1):
    print(f"  {rank}. {config_labels[config_order.index(name)]:30s} IGD = {val:.4f}")